In [1]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"):
    print("📁", root)
    for d in dirs:
        print("   -", d)

📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset
   - valid
   - test
   - train
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/valid/nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/test/nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train
   - wildfire
   - nowildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train/wildfire
📁 /kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset/train/nowildfire


# 1. Configuration
## Import Libraries
This includes Tensorflow Keras items for deeplearning framework, CNN backbone (ResNet50), model construction, training control and image loading pipelines

In [2]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.data.experimental import ignore_errors
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.resnet50 import preprocess_input

print("Libraries installed successfully!")

2026-03-19 22:34:31.017237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773959671.038943     768 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773959671.045570     768 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773959671.063224     768 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773959671.063256     768 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773959671.063259     768 computation_placer.cc:177] computation placer alr

Libraries installed successfully!


## General Parameters
Parameters that control data loading, model building and training.

In [3]:
BASE_DIR = "/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE


# 2. Data Loading

In [4]:
train_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "train"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=True,
                seed=SEED
                )
val_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "valid"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=False
                )
test_ds = image_dataset_from_directory(
                os.path.join(BASE_DIR, "test"),
                batch_size=BATCH_SIZE,
                image_size=IMAGE_SIZE,
                label_mode="binary",
                shuffle=False
                )

Found 30250 files belonging to 2 classes.


I0000 00:00:1773959681.228656     768 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 6300 files belonging to 2 classes.
Found 6300 files belonging to 2 classes.


# Exploratory Data Analysis (EDA)
To be included

# Data Augmentation

## Ignore Errors / Prefetch
This skips bad files instead of crashing. ***In a future edit, this will be replaced with EDA and proper preprocessing.***

In [5]:
train_ds = train_ds.apply(ignore_errors())
val_ds = val_ds.apply(ignore_errors())
test_ds = test_ds.apply(ignore_errors())

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


## Augmentation
These small, random transformations will result in a more robust training, allowing the model to adapt to many different ways a wildfire may look.

In [6]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(.05),
    tf.keras.layers.RandomZoom(.1),
    tf.keras.layers.RandomContrast(.1),
])

# Build the Model

***NOTE: Add learning rate schedule (eg. lr_callback = tf.keras.callbacks.ReduceLROnPlateau(--hp))***

In [7]:
# Create the Input Tensor
inputs = Input(shape=(224, 224, 3))

#Load ResNet50 Without Classifier Head
base_model = ResNet50(weights="imagenet", include_top=False, input_tensor=inputs)

#Build Custom Classifier Head
x = data_augmentation(inputs)
x = preprocess_input(x)

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(.3)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(.2)(x)
outputs = Dense(1, activation="sigmoid")(x)

#Fine-tune Top Layers
base_model.trainable = True

for layer in base_model.layers[:-30]: #This freezes the backbone layers to preserve general visual features
    layer.trainable = False

model=Model(inputs=inputs, outputs=outputs)
model.compile(                               #This trains the top layers with small learning rate
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# model.summary()

# Train the Model

In [8]:
callbacks = [
    EarlyStopping(patience=3, 
                  verbose=1, 
                  restore_best_weights=True
                 ),
    ModelCheckpoint("best_resnet50_model.keras",
                   save_best_only=True,
                   verbose=1),
    ReduceLROnPlateau(monitor="val_loss",
                     factor=.2,
                     patience=2,
                     verbose=1)    
]

model_fit = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10


I0000 00:00:1773959695.441304     834 service.cc:152] XLA service 0x7fec4c002390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773959695.441336     834 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773959697.909075     834 cuda_dnn.cc:529] Loaded cuDNN version 91002


      2/Unknown 20s 69ms/step - accuracy: 0.5781 - loss: 0.9110

I0000 00:00:1773959704.973914     834 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


    342/Unknown 43s 66ms/step - accuracy: 0.8157 - loss: 0.4007

...

944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9947 - loss: 0.0164
Epoch 4: val_loss did not improve from 0.05088
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9947 - loss: 0.0164 - val_accuracy: 0.9840 - val_loss: 0.0571 - learning_rate: 1.0000e-05
Epoch 5/10
352/945 ━━━━━━━━━━━━━━━━━━━━ 39s 66ms/step - accuracy: 0.9986 - loss: 0.0081

2026-03-19 22:40:29.867640: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 46/350


944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9985 - loss: 0.0077
Epoch 5: val_loss did not improve from 0.05088

Epoch 5: ReduceLROnPlateau reducing learning rate to 1.9999999494757505e-06.
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9985 - loss: 0.0077 - val_accuracy: 0.9854 - val_loss: 0.0566 - learning_rate: 1.0000e-05
Epoch 6/10
345/945 ━━━━━━━━━━━━━━━━━━━━ 39s 66ms/step - accuracy: 0.9991 - loss: 0.0044

2026-03-19 22:41:40.119184: E tensorflow/core/lib/jpeg/jpeg_mem.cc:329] Premature end of JPEG data. Stopped at line 46/350


944/945 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.9991 - loss: 0.0044
Epoch 6: val_loss did not improve from 0.05088
945/945 ━━━━━━━━━━━━━━━━━━━━ 71s 75ms/step - accuracy: 0.9991 - loss: 0.0044 - val_accuracy: 0.9863 - val_loss: 0.0548 - learning_rate: 2.0000e-06
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.


# Next Steps
- Evaluation
- Visualization / Transparency
- Summarization
- Annotation